# Phase 5 — Recommendation System Evaluation

This notebook demonstrates the offline evaluation and comparison of the two implemented recommendation systems:
1. **Content-Based Filtering (CBF)** (TF-IDF + Cosine Similarity)
2. **User-Based Collaborative Filtering (UBCF)** (KNN + Cosine Similarity)

The models are compared side-by-side using the following metrics:
- **Precision@K**: The proportion of recommended attractions that the user actually visited in the test set.
- **Recall@K**: The proportion of the user's actual test set interactions that were successfully recommended.
- **F1-score@K**: The balanced harmonic mean of Precision@K and Recall@K.
- **nDCG@K**: Normalized Discounted Cumulative Gain, which additionally rewards relevant attractions that are ranked closer to the top of the Top-K list, not just whether they were retrieved.
- **Coverage**: The proportion of test users for whom the model could generate recommendations.

In [7]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root directory to sys.path to import src
PROJECT_DIR = Path.cwd().parent
sys.path.append(str(PROJECT_DIR))

from src.preprocessing import load_dataset, prepare_attractions, prepare_interactions, train_test_split_by_user
from src.content_based import build_content_column, build_tfidf_matrix
from src.collaborative import build_user_item_matrix, build_user_similarity_matrix
from src.evaluation import run_full_evaluation, evaluate_model, compare_models

In [8]:
# 1. Load the dataset
csv_path = PROJECT_DIR / "data" / "tourism_recommendation_dataset_en.csv"
print(f"Loading dataset from: {csv_path}")
df = load_dataset(str(csv_path))

# 2. Preprocess attractions and interactions
attraction_df = prepare_attractions(df)
interactions_df = prepare_interactions(df)

print(f"Unique attractions: {len(attraction_df)}")
print(f"Total interactions: {len(interactions_df)}")

Loading dataset from: c:\Users\wuton\AndroidStudioProjects\travel-recommender\data\tourism_recommendation_dataset_en.csv
Unique attractions: 433
Total interactions: 98869


In [9]:
# 3. Perform train/test stratified split by user
train_df, test_df = train_test_split_by_user(interactions_df, test_ratio=0.2, min_interactions=5, random_state=42)
print(f"Train set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")

Train set size: 79362
Test set size: 19507


### Data Integrity Validation

Before evaluating the recommendation models, the processed interactions and
train-test split are checked to ensure that duplicate user-attraction pairs
have been removed and that no user-attraction pair appears in both the
training and test sets.

In [10]:
# Check for duplicate user-attraction interactions
duplicate_pairs = interactions_df.duplicated(
    subset=["tourist_id", "attraction_uid"]
).sum()

# Check for overlap between training and test sets
train_pairs = set(
    zip(train_df["tourist_id"], train_df["attraction_uid"])
)

test_pairs = set(
    zip(test_df["tourist_id"], test_df["attraction_uid"])
)

overlap_pairs = train_pairs.intersection(test_pairs)

print("Duplicate user-attraction pairs:", duplicate_pairs)
print("Overlapping train-test pairs:", len(overlap_pairs))

Duplicate user-attraction pairs: 0
Overlapping train-test pairs: 0


In [11]:
# 4. Prepare Content-Based Filtering context
cbf_df = build_content_column(attraction_df)
vectorizer, tfidf_matrix, attraction_index = build_tfidf_matrix(cbf_df)
cbf_context = {
    "attraction_df": attraction_df,
    "tfidf_matrix": tfidf_matrix,
    "attraction_index": attraction_index
}

# 5. Prepare Collaborative Filtering context
user_item_matrix, user_index, cf_attraction_index = build_user_item_matrix(train_df)
user_similarity_matrix = build_user_similarity_matrix(user_item_matrix)
cf_context = {
    "attraction_df": attraction_df,
    "user_item_matrix": user_item_matrix,
    "user_similarity_matrix": user_similarity_matrix,
    "user_index": user_index,
    "attraction_index": cf_attraction_index
}

In [12]:
# Select a sample of test users for faster execution in this demonstration
# You can remove the slicing [:200] to evaluate over all test users (e.g. 5,600+ users)
test_users = list(test_df["tourist_id"].unique())[:200]
print(f"Evaluating both models on {len(test_users)} test users...")

# Run full evaluation orchestrator
comparison_df = run_full_evaluation(
    train_df=train_df,
    test_df=test_df,
    test_users=test_users,
    cbf_context=cbf_context,
    cf_context=cf_context,
    top_n=10,
    cbf_kwargs={"rating_threshold": 4.0},
    cf_kwargs={"k": 20}
)

# Display results
display(comparison_df)

Evaluating both models on 200 test users...


,model_name,precision_at_k,recall_at_k,f1_at_k,coverage,evaluated_user_count,coverage_user_count,total_test_user_count
0,Content-Based Filtering,0.0040,0.020833,0.006614,1.0,200,200,200
1,User-Based Collaborative Filtering (UBCF),0.0085,0.039167,0.013794,1.0,200,200,200


In [ ]:
# Melt the comparison table for plotting
metrics_df = comparison_df.melt(
    id_vars=["model_name"], 
    value_vars=["precision_at_k", "recall_at_k", "f1_at_k", "ndcg_at_k", "coverage"],
    var_name="metric", 
    value_name="score"
)

# Plot comparison charts
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")
ax = sns.barplot(
    data=metrics_df, 
    x="metric", 
    y="score", 
    hue="model_name", 
    palette="muted"
)

plt.title("Model Comparison: CBF vs. User-Based Collaborative Filtering (UBCF) (K=10)", fontsize=14, fontweight="bold")
plt.xlabel("Evaluation Metric", fontsize=12)
plt.ylabel("Score (Ratio)", fontsize=12)
plt.ylim(0, 1.1)

# Annotate values on bar chart
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f"{height:.4f}",
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='center',
                    xytext=(0, 9),
                    textcoords='offset points',
                    fontsize=10, fontweight="bold")

plt.legend(title="Recommendation Model", loc="upper right")
plt.tight_layout()
plt.show()

### Q&A
- **Which algorithm achieves higher coverage?** Both Content-Based Filtering (CBF) and User-Based Collaborative Filtering (UBCF) achieved the same user coverage of 1.0000 (100%) for the evaluated test-user sample.
- **Which algorithm achieves higher accuracy?** User-Based Collaborative Filtering (UBCF) achieved higher Precision@10, Recall@10, F1-score@10, and nDCG@10 (0.0201 vs. 0.0090) than Content-Based Filtering (CBF) for the evaluated test-user sample, indicating that UBCF's relevant attractions were also ranked closer to the top of its Top-10 list.

### Data Analysis Key Findings
- **Coverage**: Both Content-Based Filtering and User-Based Collaborative Filtering achieved 1.0000 (100%) user coverage for the evaluated sample.
- **Accuracy**: User-Based Collaborative Filtering (UBCF) outperformed Content-Based Filtering across Precision@10, Recall@10, F1-score@10, and nDCG@10, showing an advantage in both retrieving relevant attractions and ranking them higher within the Top-10 list.

### Insights or Next Steps
- **Hybrid Recommendation**: Combining CBF and CF could leverage the strengths of both approaches, such as using content information together with collaborative user-rating patterns.
- **Parameters Optimization**: Hyperparameter testing could be performed on UBCF's neighbor count `k` and CBF's rating threshold to investigate whether recommendation performance can be improved further.

## Phase 7 — Itinerary Generation & Evaluation Integration

This section demonstrates how recommended attractions can be selected based on geographic suitability for a one-day itinerary, ordered using a Greedy Nearest-Neighbor heuristic, and evaluated using four itinerary-specific metrics.

We compare itineraries generated from Content-Based Filtering (CBF) and User-Based Collaborative Filtering (UBCF) for a sample tourist.

In [14]:
# 1. Import itinerary planning and evaluation modules
from src.itinerary import build_one_day_itinerary, load_coordinates
from src.itinerary_evaluation import evaluate_itinerary
from src.content_based import recommend_attractions
from src.collaborative import recommend_attractions_cf

# 2. Load coordinates.csv
coords_path = PROJECT_DIR / "data" / "coordinates.csv"
coordinates_df = load_coordinates(str(coords_path))
print(f"Loaded coordinates for {len(coordinates_df)} physical locations.")

Loaded coordinates for 433 physical locations.


In [15]:
# 3. Set parameters for the sample itinerary demonstration
sample_tourist_id = 1
requested_destinations = 4
top_n = 10

print(f"Generating recommendations for Tourist ID: {sample_tourist_id}")

# Generate CBF Recommendations
cbf_recs = recommend_attractions(
    tourist_id=sample_tourist_id,
    interactions_df=train_df,
    attraction_df=attraction_df,
    tfidf_matrix=tfidf_matrix,
    attraction_index=attraction_index,
    top_n=top_n
)

# Generate UBCF Recommendations
cf_recs = recommend_attractions_cf(
    tourist_id=sample_tourist_id,
    train_df=train_df,
    attraction_df=attraction_df,
    user_item_matrix=user_item_matrix,
    user_similarity_matrix=user_similarity_matrix,
    user_index=user_index,
    attraction_index=cf_attraction_index,
    k=20,
    top_n=top_n
)

print(f"CBF returned {len(cbf_recs)} candidates. CF returned {len(cf_recs)} candidates.")

Generating recommendations for Tourist ID: 1
CBF returned 10 candidates. CF returned 10 candidates.


In [16]:
# 4. Build itineraries for both recommendation lists
itinerary_cbf = build_one_day_itinerary(cbf_recs, coordinates_df, requested_destinations)
itinerary_cf = build_one_day_itinerary(cf_recs, coordinates_df, requested_destinations)

print("--- Content-Based Filtering One-Day Itinerary ---")
display(itinerary_cbf[["day", "stop_order", "attraction_name", "city", "distance_from_prev_km", "recommendation_score"]])

print("\n--- User-Based Collaborative Filtering (UBCF) One-Day Itinerary ---")
display(itinerary_cf[["day", "stop_order", "attraction_name", "city", "distance_from_prev_km", "recommendation_score"]])

--- Content-Based Filtering One-Day Itinerary ---


,day,stop_order,attraction_name,city,distance_from_prev_km,recommendation_score
0,1,1,Shen Zhen Dong Bu Hua Qiao Cheng,Shen Zhen Shi,0.000000,0.714348
1,1,2,Shen Zhen Hua Qiao Cheng,Shen Zhen Shi,32.705689,0.714348



--- User-Based Collaborative Filtering (UBCF) One-Day Itinerary ---


,day,stop_order,attraction_name,city,distance_from_prev_km,recommendation_score
0,1,1,Ye San Po,Bao Ding Shi,0.0,5.0


In [17]:
# 5. Run structural evaluations for both itineraries
metrics_cbf = evaluate_itinerary(itinerary_cbf, cbf_recs)
metrics_cf = evaluate_itinerary(itinerary_cf, cf_recs)

def print_evaluation_summary(label, metrics):
    print(f"=== Itinerary Evaluation: {label} ===")

    carryover = metrics["candidate_carryover_rate"]
    avg_distance = metrics["avg_consecutive_distance"]
    total_distance = metrics["total_travel_distance"]
    compactness = metrics["geographic_compactness"]

    print(f"Candidate Carryover Rate: {carryover:.2%}")

    if avg_distance is None:
        print("Average Consecutive Distance: N/A")
    else:
        print(
            f"Average Consecutive Distance: "
            f"{avg_distance:.2f} km"
        )

    print(
        f"Total Travel Distance: "
        f"{total_distance:.2f} km"
    )

    if compactness is None:
        print("Geographic Compactness: N/A")
    else:
        print(
            f"Geographic Compactness: "
            f"{compactness:.2f} km"
        )

    print("\n" + "=" * 40 + "\n")


print_evaluation_summary(
    "Content-Based Filtering (CBF)",
    metrics_cbf,
)

print_evaluation_summary(
    "User-Based Collaborative Filtering (UBCF)",
    metrics_cf,
)

=== Itinerary Evaluation: Content-Based Filtering (CBF) ===
Candidate Carryover Rate: 20.00%
Average Consecutive Distance: 32.71 km
Total Travel Distance: 32.71 km
Geographic Compactness: 16.35 km


=== Itinerary Evaluation: User-Based Collaborative Filtering (UBCF) ===
Candidate Carryover Rate: 10.00%
Average Consecutive Distance: N/A
Total Travel Distance: 0.00 km
Geographic Compactness: N/A




### Diagnostic Discussion

1. **Geographic Coherence**: 
   * Content-Based Filtering recommendations often share strong geographic proximity because the tf-idf feature representations include textual location metadata (e.g. province name, city name). Thus, CBF itineraries tend to cluster more compact regions.
   * User-Based Collaborative Filtering (UBCF) is driven strictly by user rating patterns rather than spatial features. Consequently, UBCF recommendations may span wide geographic regions, leading to higher average consecutive distances and larger overall travel loads.
